<a href="https://colab.research.google.com/github/davidrpugh/data-analytics-with-polars/blob/main/ch07.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 7: Beginning Expressions

This notebook accompanies Chapter 7 of *Python Polars: The Definitive Guide* by Jeroen Janssens and Thijs Nieuwdorp. It is adapted for **Polars 2.1** and **Google Colab**.

**Expressions** are the idea at the heart of Polars. Almost everything you do with a DataFrame, such as selecting columns, creating new ones, filtering rows, aggregating, and sorting, is written with expressions. Understanding them well makes the rest of the library click.

> **Not to be confused with regular expressions.** A *regular expression* (regex) is a pattern for matching text. A *Polars expression* is a reusable description of a computation on data. A few Polars methods accept regexes (for example `pl.col("^.*or.*$")`), but the two ideas are unrelated.

By the end of this notebook you will be able to:

- recognize where expressions are used (`select`, `with_columns`, `filter`, `group_by`, `sort`)
- explain what an expression **is**: a lazy, reusable *tree of operations* that describes how to build one or more Series
- create expressions from **columns**, **literal values**, and **ranges**
- **rename** expressions, and know how keyword names and `alias` interact
- explain why expressions are the **idiomatic** way to work in Polars, and what goes wrong when you mix in Series

**How to use this notebook:** run the cells from top to bottom. Some cells *intentionally* trigger errors.

## 0. Setup

We pin Polars to the **2.1 series** so everyone sees the same behavior, using `%pip` so the package is installed into the running notebook kernel.

In [ ]:
%pip install -q "polars==2.1.*"

### Imports and helpers

- `download` fetches a data file from the book's repository **only if it isn't already there**. It writes to a temporary name and renames it when complete, and it raises an error on HTTP failures instead of saving an error page as if it were data.

In [ ]:
from datetime import date
from pathlib import Path
import urllib.request

import polars as pl
import polars.selectors as cs
from polars.testing import assert_frame_equal


print(f"Polars {pl.__version__}")

BASE_URL = "https://github.com/davidrpugh/data-analytics-with-polars/raw/refs/heads/main/data"
DATA_DIR = Path("sample_data")


def download(relative_path: str) -> Path:
    """Download BASE_URL/relative_path into DATA_DIR (unless it exists) and return the local path."""
    path = DATA_DIR / relative_path
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        temporary = path.with_name(path.name + ".tmp")
        urllib.request.urlretrieve(f"{BASE_URL}/{relative_path}", temporary)
        temporary.rename(path)
    return path

### The data

Our running example is a small table of ten fruits from around the world. (Yes, an avocado is a fruit: it's a large single-seeded berry.)

In [ ]:
FRUIT_CSV = download("fruit.csv")

fruit = pl.read_csv(FRUIT_CSV)
fruit

## Methods and Namespaces

The `pl.Expr` class, which represents an expression, has hundreds of methods. Memorizing them isn't the goal; knowing how they are **organized** is. General-purpose methods (arithmetic, comparison, `alias`, `cast`, `sum`, ...) live directly on the expression. Methods that only make sense for one kind of data live in a **namespace**, accessed with a dot:

| Namespace | For data of type | Example |
|---|---|---|
| `.str` | Strings | `pl.col("name").str.ends_with("berry")` |
| `.dt` | Dates and times | `pl.col("when").dt.year()` |
| `.cat` | Categoricals | `pl.col("size").cat.get_categories()` |
| `.list` | Lists | `pl.col("numbers").list.len()` |
| `.arr` | Arrays | `pl.col("point").arr.sum()` |
| `.struct` | Structs | `pl.col("rating").struct.field("movie")` |
| `.name` | the *name* of the output column | `pl.all().name.suffix("_new")` |
| `.meta` | the expression itself (introspection) | `expr.meta.root_names()` |

Chapter 12 covers the data-type namespaces in depth. Here is one tiny example from three of them, so you can see the pattern: *namespace first, then the method that fits the data type.*

In [ ]:
namespaces_df = pl.DataFrame(
    {
        "text": ["Polars", "pandas"],
        "when": [date(2024, 3, 1), date(2020, 5, 17)],
        "numbers": [[1, 2, 3], [4]],
    }
)

namespaces_df.select(
    shout=pl.col("text").str.to_uppercase(),
    year=pl.col("when").dt.year(),
    how_many=pl.col("numbers").list.len(),
)

## Expressions by Example

Expressions are easiest to understand when you see them work. Notice a pattern in all five examples: **what you compute (the expression) is separate from where you use it (the method)**. We cover each method in its own chapter later; the expressions are what matter here.

### Selecting columns with expressions

`select` keeps only the columns you describe (and can create new ones). We pass four different things:

- `pl.col("name")`: an expression referring to the column `name`. `pl.col` is the most common way to *start* an expression.
- `pl.col("^.*or.*$")`: a **regex**. If the argument starts with `^` and ends with `$`, Polars treats it as a pattern and selects *every column whose name matches*. Here: the two names containing "or" (`color`, `origin`).
- `pl.col("weight") / 1000`: **arithmetic** on an expression. Polars automatically turns the integer column into floats so fractions fit. The result keeps the name `weight`, which is now misleading (it's kilograms). We fix that in the next cell.
- `"is_round"`: a plain string also works as shorthand for `pl.col("is_round")`. It's convenient, but a string is *not* an expression, so you can't do arithmetic or call methods on it.

In [ ]:
fruit.select(
    pl.col("name"),
    pl.col("^.*or.*$"),
    pl.col("weight") / 1000,
    "is_round",
)

**Better:** give a computed column an honest name with a keyword (`weight_kg=...`), and select by pattern using **selectors** (Chapter 10), which are easier to read than regexes. `cs.contains("or")` selects columns whose name contains "or". We check that it matches the regex version:

In [ ]:
by_regex = fruit.select(pl.col("^.*or.*$")).columns
by_selector = fruit.select(cs.contains("or")).columns


In [ ]:
# throws error if the two sets of columns differ
assert by_regex == by_selector

In [ ]:
print(by_selector)

In [ ]:
fruit.select(
    "name",
    cs.contains("or"),
    weight_kg=pl.col("weight") / 1000,
    is_round=pl.col("is_round")
)

### Creating new columns with expressions

`with_columns` **adds** columns (or replaces ones with the same name) while keeping all existing columns.

- `pl.lit(True)` creates an expression from a **literal value** (`lit` is short for "literal"). One value is repeated for every row. `.alias("is_fruit")` names it.
- `pl.col("name").str.ends_with("berry")` uses the `.str` namespace and produces a Boolean column. Here we name it with the **keyword** form, `is_berry=...`.

Both ways of naming do the same job; we generally prefer the keyword form because it's shorter and clearer.

In [ ]:
fruit.with_columns(
    pl.lit(True).alias("is_fruit"),
    is_berry=pl.col("name").str.ends_with("berry"), # prefer keyword form
)

### Filtering rows with expressions

`filter` keeps the rows where the expression is `True`. We combine two conditions with `&` (logical AND). The **parentheses matter**: in Python, `&` binds more tightly than `>`, so `pl.col("weight") > 1000 & ...` would be parsed wrongly. A column that is already Boolean (`is_round`) can be used as a condition directly.

Passing the conditions as *separate arguments* means the same thing as `&`:

In [ ]:
heavy_and_round = fruit.filter((pl.col("weight") > 1000) & pl.col("is_round"))
heavy_and_round

### Aggregating with expressions

Aggregating means *grouping* rows and *summarizing* each group. Expressions appear in **two** places here:

1. in `group_by`, to **define the groups**. Here the group is the last word of the `origin` column (`"South America"` becomes `"America"`, so the continent). We name it with `alias("continent")`
2. in `agg`, to **summarize each group**: `pl.len()` counts the rows, and `.mean()` averages the weights

Groups are processed in parallel, so their order isn't guaranteed. We `sort` the result to get a stable, readable output.

In [ ]:
(
    fruit.group_by(pl.col("origin").str.split(" ").list.last().alias("continent"))
    .agg(
        n_fruit=pl.len(),
        average_weight=pl.col("weight").mean(),
    )
    .sort("continent")
)

### Sorting rows with expressions

You can sort by an expression that **isn't a column** in the DataFrame. Here we sort by the *length of each fruit's name*, longest first, without adding a length column. Several names have the same length, so we add `"name"` as a tie-breaker (ascending), which makes the order fully deterministic. `descending` takes one Boolean per sort key.

`str` has two length methods, because "length" is ambiguous for text: `len_chars()` counts **characters**, and `len_bytes()` counts **bytes** in the UTF-8 encoding. For plain English letters they agree. For other characters they don't (a character can take 1 to 4 bytes). Use `len_chars()` when you mean "how many letters", and `len_bytes()` when you need speed (it's much cheaper to compute).

In [ ]:
fruit.sort(pl.col("name").str.len_chars(), "name", descending=[True, False])

In [ ]:
pl.DataFrame({"text": ["naive", "naïve", "日本"]}).select(
    "text",
    chars=pl.col("text").str.len_chars(),
    bytes=pl.col("text").str.len_bytes(),
)

## The Definition of an Expression

Now that you've seen expressions at work, here is the definition:

> An **expression** is a **tree of operations** that **describes** how to **construct** **one or more Series**.

Let's unpack each part.

- **Series.** A Series is a named column of values with one data type. An expression ends up producing Series.
- **Tree of operations.** An expression can be a single operation (`pl.col("weight")`), a chain (`pl.col("weight").mean().round(1)`), or a real tree with branches (`pl.col("a") + pl.col("b") * 2`).
- **Describe.** An expression is *only a description*. Creating one computes nothing, just as writing a recipe doesn't cook the meal. Methods like `select` and `group_by` are the "cooks" that execute the recipe.
- **Construct.** The Series an expression builds doesn't always become a new column. `filter` uses it only to decide which rows to keep, and a bare `pl.col("x")` just refers to an existing column.
- **One or more.** A single expression can describe *several* Series. `pl.all()` means "every column", so `pl.all().mul(10)` works on all of them at once.

Let's look at an expression without running it:

In [ ]:
weight_kg = (pl.col("weight") / 1000).alias("weight_kg")

print(type(weight_kg))
print(weight_kg)

Printing shows a *description*. No data was touched. Only now, when we hand the expression to a method that runs it, is a Series constructed:

In [ ]:
fruit.select(weight_kg)

**One expression, several Series.** `pl.all()` refers to all columns, `mul(10)` multiplies each by 10, and `name.suffix(...)` renames each output so the new columns don't clash with the originals:

In [ ]:
many_outputs = pl.all().mul(10).name.suffix("_times_10")

pl.DataFrame({"a": [1, 2, 3], "b": [0.4, 0.5, 0.6]}).with_columns(many_outputs)

### Peeking inside an expression with `meta`

The `meta` namespace lets you inspect an expression itself. (Chapter 18 covers it in depth.) A few examples; they are guarded because this part of the API is more likely to change between versions.

- `has_multiple_outputs()`: could this expression produce more than one Series?
- `root_names()`: which existing columns does it start from?
- `output_name()`: what will the resulting column be called?
- `tree_format()`: draw the tree of operations

In [ ]:
print("has_multiple_outputs:", many_outputs.meta.has_multiple_outputs())
print("has_multiple_outputs (weight_kg):", weight_kg.meta.has_multiple_outputs())
print("root_names:", weight_kg.meta.root_names())
print("output_name:", weight_kg.meta.output_name())

In [ ]:
print(weight_kg.meta.tree_format(return_as_string=True))

Whether `pl.all()` *actually* produces several Series depends on the DataFrame it's applied to: on a DataFrame with a single column it constructs just one.

## Properties of Expressions

Because an expression is a description, it has several useful properties.

| Property | What it means | Why it's useful |
|---|---|---|
| **Lazy** | Creating an expression computes nothing | Everything else follows from this |
| **Function- and data-dependent** | The method that runs it decides what happens to the result; the data decides the type and length | One expression, many uses |
| **Reusable** | It's an ordinary Python object you can store in a variable | Define a rule once, use it everywhere |
| **Expressive** | One expression can work on many columns and compose operations freely | Concise, readable code |
| **Efficient** | Polars can analyze and optimize an expression *before* running it, and multiple expressions in one call run **in parallel** | Speed |

**Function dependent and reusable.** Here is one expression, `is_orange`, used with three different methods. In `with_columns` it becomes a new column, in `filter` it picks rows, and in `group_by` it defines the groups:

In [ ]:
is_orange = (pl.col("color") == "orange").alias("is_orange")

fruit.with_columns(is_orange)

In [ ]:
fruit.filter(is_orange)

In [ ]:
fruit.group_by(is_orange).len().sort("is_orange")

**Data dependent and reusable.** The same expression works on a *completely different DataFrame*, as long as it has a `color` column. The data determines the length and the type of the Series that comes out:

In [ ]:
flowers = pl.DataFrame(
    {
        "name": ["Tiger lily", "Blue flag", "African marigold"],
        "latin": ["Lilium columbianum", "Iris versicolor", "Tagetes erecta"],
        "color": ["orange", "purple", "orange"],
    }
)

flowers.filter(is_orange)

**Efficient.** Because expressions are descriptions, Polars can look at all the expressions you pass to a single call, optimize them, and run independent ones **at the same time** on different CPU cores. (One consequence: expressions in the same `with_columns` call can't refer to *each other's* results. Chapter 10 shows how to handle that.)


## Creating Expressions

Every expression starts with a *first operation*. There are three main starting points: existing **columns**, literal **values**, and **ranges**.

### From existing columns

The most common start is a reference to a column you already have, with `pl.col`. It accepts names, regexes, and data types. Here we call `.columns` on the result to see which columns each expression selects. A single name:

In [ ]:
fruit.select(pl.col("color")).columns

If the column doesn't exist, Polars raises an error. Let's see the message in your version:

In [ ]:
fruit.select(pl.col("is_smelly")).columns

**Several columns at once:**

- a regex, which must start with `^` and end with `$`
- `pl.all()`, shorthand for "every column"
- a **data type**: all columns of that type
- several names or types, passed as separate arguments

In [ ]:
print("regex:        ", fruit.select(pl.col("^.*or.*$")).columns)
print("all:          ", fruit.select(pl.all()).columns)
print("String type:  ", fruit.select(pl.col(pl.String)).columns)
print("Boolean/Int64:", fruit.select(pl.col(pl.Boolean, pl.Int64)).columns)
print("two names:    ", fruit.select(pl.col("name", "color")).columns)

(`pl.col(["name", "color"])`, with a list, also works. Passing the names as separate arguments is simpler.) For selecting columns by their *properties* (such as "all numeric columns"), **column selectors** are the recommended tool, and Chapter 10 covers them.


### From literal values

`pl.lit(value)` creates an expression from a plain Python value. We run these with `pl.select`, which starts from an *empty* DataFrame, so the expression produces just one value. The column is literally called `literal` until you rename it, either with `alias` or with a keyword:

In [ ]:
literal_demo = pl.select(pl.lit(42))

In [ ]:
print(literal_demo.schema)

In [ ]:
literal_demo

In [ ]:
print(pl.select(pl.lit(42).alias("answer")).columns)

In [ ]:
print(pl.select(answer=pl.lit(42)).columns)

**Broadcasting.** When you use a *single* literal value inside a DataFrame that has rows, Polars repeats it so the column has the right length. (All columns in a DataFrame must have the same length.)

In [ ]:
fruit.with_columns(planet=pl.lit("Earth"))

That only works for **one** value. If you give several values, but not as many as there are rows, there is no sensible way to line them up, and Polars raises an error:

In [ ]:
fruit.with_columns(row_is_even=pl.lit(pl.Series([False, True])))

Passing a plain Python *list* to `pl.lit` does something different: it creates a **List-typed value** that is repeated for every row, so each row contains the same two-element list:

In [ ]:
fruit.with_columns(row_is_even=pl.lit([False, True])).select("name", "row_is_even").head(3)

To repeat a value a specific number of times, use `pl.repeat`. The number of repeats must match the number of rows if you add it to an existing DataFrame:

In [ ]:
pl.select(
    umbrella=pl.repeat("Ella", 3),
    zeros=pl.repeat(0.0, 3),
    ones=pl.repeat(1.0, 3),
)

In [ ]:
fruit.with_columns(planet=pl.repeat("Earth", 9))  # fruit has 10 rows

### From ranges

Polars has functions that generate sequences of integers, dates, times, and datetimes:

| Function | Produces |
|---|---|
| `pl.int_range(...)` | one integer range (a column) |
| `pl.int_ranges(...)` | *one range per row* (a column of lists) |
| `pl.date_range(...)`, `pl.datetime_range(...)`, `pl.time_range(...)` | one range of dates / datetimes / times |
| `pl.date_ranges(...)`, `pl.datetime_ranges(...)`, `pl.time_ranges(...)` | one range per row |

The singular form builds a *column*. The plural form takes start and end *columns* and builds, for each row, a whole range stored as a **List**. (The book also uses the alias `pl.arange`; prefer the documented name `pl.int_range`.)

In [ ]:
(
    pl.select(
        start=pl.int_range(0, 5),
        end=pl.int_range(0, 10, 2).pow(2),
    )
    .with_columns(int_range=pl.int_ranges("start", "end"))
    .with_columns(range_length=pl.col("int_range").list.len())
)

The same idea for dates. The first column holds six consecutive dates; the second repeats a single end date six times; `datetime_ranges` then builds, for each row, a list of **hourly** timestamps from the start date up to (and excluding) the end date. We show only the length of each list, since the lists themselves are long.

In [ ]:
(
    pl.select(
        start=pl.date_range(pl.date(1985, 10, 21), pl.date(1985, 10, 26)),
        end=pl.repeat(pl.date(1985, 10, 27), 6),
    )
    .with_columns(range=pl.datetime_ranges("start", "end", interval="1h"))
    .with_columns(n_hours=pl.col("range").list.len())
    .select("start", "end", "n_hours")
)

### Other functions that create expressions

There are many more starting points than we can cover here. Two you will meet soon:

- **`pl.len()`** counts rows. You've used it in `agg`. We cover it with aggregation in Chapter 13.
- **`pl.element()`** refers to *each element* of a list, inside `list.eval`. This is how you run an expression on the *inside* of a List column (Chapter 12):

In [ ]:
pl.DataFrame({"numbers": [[1, 2, 3], [4, 5]]}).with_columns(
    doubled=pl.col("numbers").list.eval(pl.element() * 2)
)

## Renaming Expressions

The name of an expression decides the name of the column it produces. You rename for several reasons: to describe a computed column better, to avoid **duplicate** column names (Polars requires unique names), to clean up messy names, or to change a default name.

Good names matter as much as good variable names. We recommend `snake_case`: lowercase words separated by underscores.

There are two ways to set a name directly:

- `expr.alias("new_name")`: the name travels *with the expression*, wherever it's used
- `new_name=expr`: a **keyword**, where the name is local to the method call

If you use both on the same expression, **the keyword wins**:

In [ ]:
fruit.select(
    pl.col("weight").alias("from_alias"),
    from_keyword=pl.col("weight").alias("ignored"),
).columns

Keyword names must be valid Python identifiers (no spaces, can't start with a digit, can't be a reserved word like `class`) and, once you use one keyword argument, everything after it must be a keyword too. When those rules get in the way, use `alias`.

The `name` namespace has methods that change the name of an expression's output based on the *existing* name, which is useful when one expression produces *many* columns:

| Method | Effect |
|---|---|
| `name.keep()` | keep the original name (undoing an alias) |
| `name.prefix(...)`, `name.suffix(...)` | add text before or after |
| `name.to_lowercase()`, `name.to_uppercase()` | change case |
| `name.map(function)` | compute the new name with a Python function |

Here they are on a DataFrame with messy names:

In [ ]:
naming_df = pl.DataFrame({"text": "value", "An integer": 5040, "BOOLEAN": True})
naming_df

In [ ]:
naming_df.select(
    pl.col("text").name.to_uppercase(),
    pl.col("An integer").alias("int"),
    pl.col("BOOLEAN").name.to_lowercase(),
)

## Expressions Are Idiomatic

You can mix plain Series into Polars code. Everything above can be written, pandas-style, by pulling out columns with brackets:

In [ ]:
with_series = fruit.filter((fruit["weight"] > 1000) & fruit["is_round"])
with_expressions = fruit.filter((pl.col("weight") > 1000) & pl.col("is_round"))

assert_frame_equal(with_series, with_expressions)

In [ ]:
print(pl.select(answer=pl.lit(42)).columns)

Same answer, so why avoid it? Two reasons:

1. **Optimization.** `fruit["weight"] > 1000` is computed *immediately*, before the filter method even runs. Polars only sees the finished result, so it can't analyze or reorder the work. It also can't run the two conditions in parallel.
2. **Correctness.** `fruit["name"]` always refers to the *original* 10-row DataFrame. If you use it *after* the data has been filtered down, the lengths no longer match.

Compare the query plans. In the expression version the conditions are part of the plan, where the optimizer can work with them. In the Series version they have already been turned into fixed values:

In [ ]:
lazy_expressions = fruit.lazy().filter((pl.col("weight") > 1000) & pl.col("is_round"))

In [ ]:
print("With expressions:")
print(lazy_expressions.explain())

In [ ]:
fruit.lazy().filter((fruit["weight"] > 1000) & fruit["is_round"]).explain()

Now the second problem. A lazy chain that filters the DataFrame down to 2 rows and *then* adds a column built from the original 10-row Series. Expressions, by contrast, always refer to the data as it is *at that step*:

In [ ]:
(
    fruit.lazy()
         .filter((pl.col("weight") > 1000) & pl.col("is_round"))
         .with_columns(is_berry=pl.col("name").str.ends_with("berry"))
         .collect()
)

In [ ]:
(
    fruit.lazy()
         .filter((fruit["weight"] > 1000) & fruit["is_round"])
         .with_columns(is_berry=fruit["name"].str.ends_with("berry"))
         .collect()
)

The first works; the second fails because `fruit["name"]` still has 10 values while the filtered DataFrame has 2. Your version may report the problem differently (or earlier), which is why we print what happens.

**The rule of thumb:** inside Polars methods, build expressions with `pl.col(...)`; don't reach for `df["column"]`. Being lazy pays off in Polars.

## Takeaways

- Expressions are **reusable, lazy building blocks**: an expression describes a computation and runs only when a method executes it.
- An expression is a **tree of operations** that describes how to construct **one or more Series**.
- The same expression behaves differently depending on **where** it is used (`select`, `with_columns`, `filter`, `group_by`, `sort`) and on **which data** it meets.
- Start expressions from **columns** (`pl.col`, `pl.all`), **literals** (`pl.lit`, `pl.repeat`), or **ranges** (`pl.int_range`, `pl.date_range`, ...).
- A single literal is **broadcast** to the length of the DataFrame, but several values must match the number of rows.
- Name expressions with `alias` or a keyword (the keyword wins), and use the `name` namespace to rename many columns at once.
- Use **expressions, not Series brackets**, so Polars can optimize, parallelize, and keep your pipeline correct.
- Behaviors described in a book can change between versions. **Run the code and look.**

### Next up

In Chapter 8 we **continue** expressions: the many methods you can chain after the first operation.



### Try it yourself


1. Create a column `weight_class` in `fruit` that says `"light"` when `weight` is under 100 grams and `"heavy"` otherwise. (Hint: Chapter 9 introduces `pl.when`; for now, try a Boolean column from a comparison.)

2. Use `pl.col(pl.Int64)` with `.mul(2).name.suffix("_doubled")` to double every integer column of `fruit` in a single expression. Check with `meta.has_multiple_outputs()`.

3. Write an expression for "the first word of the fruit's origin" and use it three ways: in `with_columns`, in `filter`, and in `group_by`.

4. Build a DataFrame with one row per day of March 2024 using `pl.date_range`, then add a column for the day of the week with the `dt` namespace.

5. Use `pl.int_ranges` to produce, for each row of a small DataFrame, a list of the numbers from 1 up to a value in another column.